# Portfolio Optimization

This notebook covers `abaquant.portfolio`: allocation families
(mean-variance, risk-based, downside-risk), the Markowitz efficient
frontier, Monte Carlo portfolio clouds, a compact backtest, and historical
stress testing.

**Sections:**
1. Build a `PortfolioAllocator`
2. Allocation families
3. Efficient frontier and risk metrics
4. Backtest and stress tests
5. Visualizations


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
import numpy as np
import pandas as pd

from abaquant.portfolio.backtesting import run_backtest
from abaquant.portfolio.efficient_frontier import markowitz_frontier, monte_carlo_portfolios
from abaquant.portfolio.optimization import PortfolioAllocator
from abaquant.portfolio.risk_metrics import compute_all_metrics, portfolio_returns
from abaquant.portfolio.stress_testing import run_all_scenarios
from abaquant.visualization import VisualizationError

### A small deterministic price/return panel


In [ ]:
def sample_prices() -> pd.DataFrame:
    dates = pd.date_range("2025-01-02", periods=36, freq="B")
    trend = np.linspace(0.0, 1.0, len(dates))
    seasonal = np.sin(np.linspace(0.0, 4.0 * np.pi, len(dates)))
    return pd.DataFrame(
        {
            "ALPHA": 100.0 + 9.0 * trend + 1.8 * seasonal,
            "BETA": 82.0 + 6.0 * trend - 1.2 * seasonal,
            "GAMMA": 54.0 + 3.0 * trend + 0.7 * np.cos(np.linspace(0.0, 3.0 * np.pi, len(dates))),
        },
        index=dates,
    )

def sample_returns() -> pd.DataFrame:
    return sample_prices().pct_change().dropna()

prices = sample_prices()
returns = sample_returns()
returns.head()

## 1. Build a `PortfolioAllocator`

The allocator groups three families of allocation strategies:
`mean_variance`, `risk_based`, and `downside_risk`.


In [ ]:
allocator = PortfolioAllocator(returns, annual_risk_free_rate=0.02)

## 2. Allocation families

Compare equal weight, minimum variance, maximum Sharpe, risk parity,
inverse volatility, and minimum-CVaR allocations on the same universe.


In [ ]:
allocations = {
    "equal_weight": allocator.mean_variance.equal_weight(),
    "minimum_variance": allocator.mean_variance.minimum_variance(),
    "maximum_sharpe": allocator.mean_variance.maximum_sharpe(),
    "risk_parity": allocator.risk_based.risk_parity(),
    "inverse_volatility": allocator.risk_based.inverse_volatility(),
    "minimum_cvar": allocator.downside_risk.minimum_cvar(alpha=0.05),
}

summary = {}
for name, result in allocations.items():
    weights = getattr(result, "weights", result)
    first_weight = float(weights.iloc[0]) if hasattr(weights, "iloc") else float(np.asarray(weights)[0])
    summary[f"{name}_alpha_weight"] = first_weight

for key, value in summary.items():
    print(f"{key:32s}: {value:.4f}")

## 3. Efficient frontier and risk metrics

Trace the constrained Markowitz efficient frontier, sample a random
portfolio cloud, and compute a full performance/risk summary for one
fixed-weight portfolio.


In [ ]:
mean_returns = returns.mean() * 252
covariance = returns.cov() * 252
frontier = markowitz_frontier(mean_returns, covariance, n_points=8)
cloud = monte_carlo_portfolios(mean_returns, covariance, n_portfolios=250, rf=0.02, seed=42)

weights = np.array([0.4, 0.35, 0.25])
portfolio_series = portfolio_returns(returns, weights)
metrics = compute_all_metrics(portfolio_series, rf=0.02)

print(f"Frontier points:   {len(frontier)}")
print(f"Cloud portfolios:  {len(cloud)}")
print(f"Sharpe ratio:      {metrics['Sharpe Ratio']:.4f}")
print(f"Max drawdown:      {metrics['Max Drawdown']:.4f}")

In [ ]:
frontier.head()

## 4. Backtest and stress tests

Run a simple rebalanced backtest on prices, then apply predefined
historical stress scenarios to a fixed-weight portfolio. For the richer,
object-oriented backtesting API (rebalance schedules, transaction costs,
benchmarks, rolling metrics), see notebook **19 — Portfolio Backtesting**.


In [ ]:
backtest = run_backtest(
    prices, strategy_name="equal_weight", rebalance_freq="monthly", lookback_days=8
)
stress_weights = pd.Series([0.4, 0.35, 0.25], index=prices.columns)
stress = run_all_scenarios(prices, stress_weights)

print(f"Backtest object created: {backtest is not None}")
print(f"Stress scenarios evaluated: {len(stress)}")

## 5. Visualizations

Allocation weights, cumulative-return path, and the asset-correlation heatmap.


In [ ]:
try:
    max_sharpe_weights = allocations["maximum_sharpe"]
    figures = {
        "weights": allocator.visualize(weights=max_sharpe_weights, chart="weights"),
        "cumulative_returns": allocator.visualize(
            weights=max_sharpe_weights, chart="cumulative_returns"
        ),
        "correlation": allocator.visualize(chart="correlation"),
    }
    print(f"Created {len(figures)} figures: {list(figures)}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

`PortfolioAllocator` composes three allocation philosophies behind one
facade. Combine it with `.backtest()`, `.scenario_analysis()`, and
`.report()` for a full research-to-report workflow — see notebooks
**19 — Portfolio Backtesting**, **14 — Scenario Analysis**, and
**21 — Exportable Reports**.
